# Phase 3 — recovery: re-train the validation-selected family winners (weights were lost)
The previous GPU session ended (Colab free-tier usage limit) before Google Drive finished uploading the trained weights.
This notebook re-trains ONLY the family winners chosen on validation, with their exact saved configurations (no new
hyperparameters), one family at a time: train (best epoch chosen on validation by early stopping, as before) -> score
`test.csv` once -> model ZIP + results ZIP to Drive -> `drive.flush_and_unmount()` + remount -> verify the ZIPs on Drive
(size, CRC, required files) -> only then the next family. Re-running resumes: verified families are skipped.
**Run all** on a GPU runtime and allow Google Drive. Model C: Skipped — no real conversation_id/message_order available.

In [ ]:
import sys, json, hashlib, zipfile, shutil
from pathlib import Path
from datetime import datetime
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required: select an L4 / A100 / T4 runtime, then run all again.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | VRAM {props.total_memory / 2**30:.1f} GiB | CUDA (torch build) {torch.version.cuda} | torch {torch.__version__}")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import files
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE = Path("/content/drive/MyDrive/NLP_sentiment")
    DRIVE.mkdir(parents=True, exist_ok=True)
    print("Google Drive enabled:", DRIVE)
except Exception as exc:
    DRIVE = Path("/content/NLP_sentiment")
    DRIVE.mkdir(parents=True, exist_ok=True)
    print("Google Drive unavailable; using local Colab storage:", exc)
PKG_ZIP = DRIVE / "transformer_package.zip"
if not PKG_ZIP.exists():
    print("Upload transformer_package.zip from D:/NLP/sentiment_app/artifacts/colab_package_phase3")
    up = files.upload()
    PKG_ZIP.write_bytes(up[next(iter(up))])
PKG = Path("/content/pkg"); shutil.rmtree(PKG, ignore_errors=True)
with zipfile.ZipFile(PKG_ZIP) as z: z.extractall(PKG)
manifest = json.loads((PKG / "MANIFEST.json").read_text())
bad = [n for n, h in manifest.items() if hashlib.sha256((PKG / n).read_bytes()).hexdigest() != h]
if bad: raise SystemExit(f"Package files do not match MANIFEST.json: {bad}")
print(f"All {len(manifest)} package files verified. Package: {(PKG / 'VERSION.txt').read_text() if (PKG / 'VERSION.txt').exists() else 'v1'}")

In [ ]:
!pip -q install datasets accelerate sentencepiece protobuf   # only installs what is missing

In [ ]:
import shutil
from google.colab import drive
sys.path.insert(0, str(PKG))
import colab_runner
RUNS2 = Path("/content/rerun_runs"); RUNS2.mkdir(parents=True, exist_ok=True)   # work on the VM disk
OUTD = DRIVE / "phase3_rerun"; OUTD.mkdir(parents=True, exist_ok=True)          # only verified ZIPs go to Drive
runner = colab_runner.Runner(PKG / "data", RUNS2)
WINNERS = json.loads(r'''[
 {
  "experiment_id": "indicbert_lr2e-05_ce_ep3p2_bs16x1_wd0.01_wu0.1_len128",
  "model": "ai4bharat/IndicBERTv2-MLM-only",
  "learning_rate": 2e-05,
  "epochs_max": 3,
  "max_length": 128,
  "loss": "ce",
  "focal_gamma": null,
  "class_weights": null,
  "class_weight_scheme": null,
  "lang_tokens": false,
  "sampler": "random",
  "text_variant": "v1_current",
  "scheduler": "linear",
  "freeze_mode": null,
  "frozen_embeddings": false,
  "batch_size": 16,
  "grad_accum": 1,
  "effective_batch_size": 16,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "label_smoothing": 0.0,
  "freeze_lower_layers_epoch1": 0,
  "early_stopping_patience": 2,
  "metric_for_best_model": "macro_f1",
  "seed": 42,
  "fp16": true,
  "no_fp16_flag": false,
  "group_by_length": true,
  "input_col": "cleaned_text",
  "input_prefix": "",
  "resumed_from_checkpoint": null,
  "input": "cleaned_text",
  "best_epoch": 3.0,
  "epochs_run": 3.0,
  "train_seconds": 1518.1,
  "best_checkpoint": "epoch 3 (/content/checkpoints/checkpoint-13227); intermediate checkpoints deleted, best weights kept per model family by colab_runner",
  "still_improving_at_last_epoch": true,
  "overfitting_signal": null,
  "trainer_best_val_macro_f1": 0.7504865594033877,
  "device": "Tesla T4",
  "split_sha256": {
   "train": "e0835f566132bda8355b32401c23026ca0037a450b245140f4d02bbf48ca3fcb",
   "validation": "427fe31b3b1873045c12423de5c9a6469968840a8506c452aef8f5e08e027d7e",
   "test": "935b3185c808c840e672654e770eeda4d70bfe1ac1c2fbd186a308c61eeef468"
  },
  "smoke_test": false,
  "test_used": false
 },
 {
  "experiment_id": "xlmr_lr1e-05_ce_ep3p2_bs16x1_wd0.01_wu0.1_len128",
  "model": "xlm-roberta-base",
  "learning_rate": 1e-05,
  "epochs_max": 3,
  "max_length": 128,
  "loss": "ce",
  "focal_gamma": null,
  "class_weights": null,
  "class_weight_scheme": null,
  "lang_tokens": false,
  "sampler": "random",
  "text_variant": "v1_current",
  "scheduler": "linear",
  "freeze_mode": null,
  "frozen_embeddings": false,
  "batch_size": 16,
  "grad_accum": 1,
  "effective_batch_size": 16,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "label_smoothing": 0.0,
  "freeze_lower_layers_epoch1": 0,
  "early_stopping_patience": 2,
  "metric_for_best_model": "macro_f1",
  "seed": 42,
  "fp16": true,
  "no_fp16_flag": false,
  "group_by_length": true,
  "input_col": "cleaned_text",
  "input_prefix": "",
  "resumed_from_checkpoint": null,
  "input": "cleaned_text",
  "best_epoch": 2.0,
  "epochs_run": 3.0,
  "train_seconds": 1487.3,
  "best_checkpoint": "epoch 2 (/content/checkpoints/checkpoint-8818); intermediate checkpoints deleted, best weights kept per model family by colab_runner",
  "still_improving_at_last_epoch": false,
  "overfitting_signal": null,
  "trainer_best_val_macro_f1": 0.7487087749347326,
  "device": "Tesla T4",
  "split_sha256": {
   "train": "e0835f566132bda8355b32401c23026ca0037a450b245140f4d02bbf48ca3fcb",
   "validation": "427fe31b3b1873045c12423de5c9a6469968840a8506c452aef8f5e08e027d7e",
   "test": "935b3185c808c840e672654e770eeda4d70bfe1ac1c2fbd186a308c61eeef468"
  },
  "smoke_test": false,
  "test_used": false
 },
 {
  "experiment_id": "muril_lr3e-05_ce_ep3p2_bs16x1_wd0.01_wu0.1_len128",
  "model": "google/muril-base-cased",
  "learning_rate": 3e-05,
  "epochs_max": 3,
  "max_length": 128,
  "loss": "ce",
  "focal_gamma": null,
  "class_weights": null,
  "class_weight_scheme": null,
  "lang_tokens": false,
  "sampler": "random",
  "text_variant": "v1_current",
  "scheduler": "linear",
  "freeze_mode": null,
  "frozen_embeddings": false,
  "batch_size": 16,
  "grad_accum": 1,
  "effective_batch_size": 16,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "label_smoothing": 0.0,
  "freeze_lower_layers_epoch1": 0,
  "early_stopping_patience": 2,
  "metric_for_best_model": "macro_f1",
  "seed": 42,
  "fp16": true,
  "no_fp16_flag": false,
  "group_by_length": true,
  "input_col": "cleaned_text",
  "input_prefix": "",
  "resumed_from_checkpoint": null,
  "input": "cleaned_text",
  "best_epoch": 3.0,
  "epochs_run": 3.0,
  "train_seconds": 1361.5,
  "best_checkpoint": "epoch 3 (/content/checkpoints/checkpoint-13227); intermediate checkpoints deleted, best weights kept per model family by colab_runner",
  "still_improving_at_last_epoch": true,
  "overfitting_signal": null,
  "trainer_best_val_macro_f1": 0.741375592036874,
  "device": "Tesla T4",
  "split_sha256": {
   "train": "e0835f566132bda8355b32401c23026ca0037a450b245140f4d02bbf48ca3fcb",
   "validation": "427fe31b3b1873045c12423de5c9a6469968840a8506c452aef8f5e08e027d7e",
   "test": "935b3185c808c840e672654e770eeda4d70bfe1ac1c2fbd186a308c61eeef468"
  },
  "smoke_test": false,
  "test_used": false
 }
]''')
WEIGHTS = ("model.safetensors", "pytorch_model.bin")
TOKENIZER = ("tokenizer.json", "sentencepiece.bpe.model", "spiece.model", "vocab.txt")
REQUIRED = ("config.json", "phase3_config.json", "tokenizer_config.json")

# the exact frozen split (the package itself was already checked file-by-file against MANIFEST.json above)
import hashlib, pandas as pd
ROWS = {"train": 70540, "validation": 15121, "test": 15124}
for split, n in ROWS.items():
    df = pd.read_csv(PKG / "data" / f"{split}.csv", encoding="utf-8-sig", low_memory=False)
    assert len(df) == n, (split, len(df))
    h = hashlib.sha256((PKG / "data" / f"{split}.csv").read_bytes()).hexdigest()
    assert all(w["split_sha256"][split] == h for w in WINNERS), f"{split}.csv differs from the winners' split"
print("SPLIT_OK", ROWS)

def check_model_zip(path, eid):
    if not path.exists():
        return "missing"
    size = path.stat().st_size
    if size < 300e6:
        return f"too small ({size / 1e6:.1f} MB)"
    try:
        with zipfile.ZipFile(path) as z:
            names = set(z.namelist())
            if z.testzip() is not None:
                return "CRC error"
            if json.loads(z.read("phase3_config.json")).get("experiment_id") != eid:
                return "wrong experiment"
    except (zipfile.BadZipFile, KeyError, OSError, ValueError) as e:      # e.g. an upload that never completed
        return f"unreadable ({type(e).__name__})"
    if not any(w in names for w in WEIGHTS):
        return "no weight file"
    if not any(t in names for t in TOKENIZER):
        return "no tokenizer file"
    missing = [r for r in REQUIRED if r not in names]
    return f"missing {missing}" if missing else "ok"

def check_results_zip(path, eid):
    if not path.exists():
        return "missing"
    try:
        with zipfile.ZipFile(path) as z:
            names = set(z.namelist())
            if z.testzip() is not None:
                return "CRC error"
    except (zipfile.BadZipFile, OSError) as e:
        return f"unreadable ({type(e).__name__})"
    need = [f"{eid}/{n}" for n in ("config.json", "validation_metrics.json", "predictions.csv", "test_metrics.json",
                                       "predictions_test.csv")]
    missing = [n for n in need if n not in names]
    return f"missing {missing}" if missing else "ok"

def persist_and_verify(mz, rz, eid):
    drive.flush_and_unmount()                # blocks until every cached file is uploaded
    drive.mount("/content/drive")            # files are now read back from Drive itself
    m, r = check_model_zip(mz, eid), check_results_zip(rz, eid)
    print("VERIFY", mz.name, m, f"{mz.stat().st_size / 1e9:.2f} GB" if mz.exists() else "", "|", rz.name, r)
    if m != "ok" or r != "ok":
        raise RuntimeError(f"Drive verification failed for {eid}: model {m}, results {r}")

for saved in WINNERS:
    cfg = colab_runner.cfg_from_saved(saved)
    eid = colab_runner.exp_id(dict(colab_runner.DEFAULT, **cfg))
    assert eid == saved["experiment_id"], (eid, saved["experiment_id"])       # exact saved configuration
    fam = cfg["family"]
    mz, rz = OUTD / f"phase3_model_{fam}.zip", OUTD / f"phase3_results_rerun_{fam}.zip"
    if check_model_zip(mz, eid) == "ok" and check_results_zip(rz, eid) == "ok":
        print("SKIP (already verified on Drive):", fam, eid)
        continue
    print("=== re-training", fam, eid, flush=True)
    shutil.rmtree(RUNS2 / eid, ignore_errors=True)       # a half-finished earlier attempt is never reused
    shutil.rmtree(RUNS2 / f"_best_model_{fam}", ignore_errors=True)
    runner.run(cfg)                                      # validation-only training; best epoch by val macro F1
    runner.final_test()                                  # test.csv once, for this family's (only) run
    v = json.loads((RUNS2 / eid / "validation_metrics.json").read_text())
    t = json.loads((RUNS2 / eid / "test_metrics.json").read_text())
    c = json.loads((RUNS2 / eid / "config.json").read_text())
    print(f"{fam}: best epoch {c['best_epoch']:g} | VAL macro F1 {v['macro_f1']:.4f} | TEST macro F1 "
          f"{t['macro_f1']:.4f} acc {t['accuracy']:.4f}")
    local_m = Path(f"/content/phase3_model_{fam}.zip")
    with zipfile.ZipFile(local_m, "w", zipfile.ZIP_STORED) as z:
        for p in sorted((RUNS2 / f"_best_model_{fam}" / "model").iterdir()):
            if p.is_file():
                z.write(p, p.name)
    local_r = Path(f"/content/phase3_results_rerun_{fam}.zip")
    with zipfile.ZipFile(local_r, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted((RUNS2 / eid).rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(RUNS2).as_posix())
        z.writestr("_selected_for_test.json", json.dumps({fam: eid}))
    assert check_model_zip(local_m, eid) == "ok", check_model_zip(local_m, eid)
    shutil.copy(local_m, mz)
    shutil.copy(local_r, rz)
    persist_and_verify(mz, rz, eid)
    print("PERSISTED", fam, eid, flush=True)
print("ALL_FAMILIES_PERSISTED")